# Notebook 01 — Process Real Data
Convert raw Cowrie logs + CIC/UNSW-NB15 into structured session DataFrames.
**Target:** 180,000 real Cowrie + 60,000 transfer = 240,000 real sessions.

In [1]:
import sys, os, json, logging, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger()

# ── Ensure project root is on path ──
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
DATA_RAW  = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
DATA_PROC.mkdir(parents=True, exist_ok=True)
# HS_SMOKE caps every source to roughly this many rows so the whole pipeline
# (01 -> 02 -> 05) can be run end-to-end in minutes and the OUTPUT checked
# against DATA_INTEGRITY.md before committing to the hours-long real build.
# Unset or 0 means the full rebuild. A smoke build is NOT a training corpus:
# dead-column checks need tens of thousands of rows to mean anything, so only
# the arithmetic-invariant results transfer from a smoke run.
SMOKE = int(os.environ.get("HS_SMOKE", "0"))
# BENIGN is 2,273,097 of CIC-IDS2017's 2,830,743 rows (80%) plus 93,000 UNSW
# "Normal". Uncapped, the build is 3.9M rows and 2.37M of them are benign --
# rows that cost DistilBERT and extraction time and then go into X_benign.npy,
# which no model consumes yet. 100k is 50x the 2,000-per-class floor and ample
# for a two-stage gate. Raise it when the gate actually needs more.
BENIGN_CAP = int(os.environ.get("HS_BENIGN_CAP", "100000"))
if SMOKE:
    print(f"*** HS_SMOKE={SMOKE}: capping every source. NOT a training corpus. ***")

print("Project root:", ROOT)

Project root: C:\Users\mahit\OneDrive\Desktop\adaptive-honeypot-ml-CAPSTONE\honeypot_dataset


## 1.1  Parse Cowrie JSON Logs

In [2]:
import re, math
from collections import defaultdict
from datetime import datetime, timezone
import pandas as pd
import numpy as np

# The real 13-day capture lives at the REPO ROOT, not under data/raw/.
# 114,490 sessions / 754,063 events. Gitignored and irreplaceable.
COWRIE_LOG_DIR = ROOT.parent / "cowrie_logs"

# Map Cowrie event types → micro-state labels
EVENT_MAP = {
    "cowrie.session.connect":       "RECON_IP_SCAN",
    "cowrie.login.failed":          "ACCESS_BRUTE_SSH",
    "cowrie.login.success":         "ACCESS_BRUTE_SSH",
    "cowrie.session.file_download": "EXEC_WGET_EXEC",
    "cowrie.session.file_upload":   "EXFIL_SCP_DATA",
    "cowrie.direct-tcpip.request":  "LATERAL_SSH_SPREAD",
}

COMMAND_PATTERNS = [
    (re.compile(r"\b(uname|hostname|id|whoami|cat /proc)\b"),     "DISC_ENV_PROBE"),
    (re.compile(r"\b(netstat|ss -|ip addr|ifconfig)\b"),           "DISC_NETSTAT_SCAN"),
    (re.compile(r"\bps\s"),                                         "DISC_PROC_ENUM"),
    (re.compile(r"find.*-perm.*4000|-perm.*-u=s"),                   "DISC_SUID_HUNT"),
    (re.compile(r"\b(wget|curl)\b.*http"),                          "EXEC_WGET_EXEC"),
    (re.compile(r"\bpython[23]?\b"),                                "EXEC_PYTHON_SCRIPT"),
    (re.compile(r"\bperl\b"),                                       "EXEC_PERL_SCRIPT"),
    (re.compile(r"bash.*-i.*>&|/dev/tcp|nc.*-e|ncat"),               "EXEC_CURL_BASH"),
    (re.compile(r"\bsudo\b"),                                       "PRIVESC_SUDO_ABUSE"),
    (re.compile(r"\bcrontab\b"),                                    "PERSIST_CRONTAB"),
    (re.compile(r"authorized_keys"),                                  "PERSIST_SSH_KEY_ADD"),
    (re.compile(r"history\s*-c|>\s*/dev/null|rm.*\.bash_history"),"EVASION_HIST_ERASE"),
    (re.compile(r"rm.*(/var/log|/var/run)"),                         "EVASION_LOG_WIPE"),
    (re.compile(r"\bscp\b"),                                        "EXFIL_SCP_DATA"),
    (re.compile(r"\btar\b.*-czf"),                                  "EXFIL_STAGING_TAR"),
]

def label_command(cmd: str) -> str:
    cmd_l = cmd.lower()
    for pattern, label in COMMAND_PATTERNS:
        if pattern.search(cmd_l):
            return label
    return "EXEC_SHELL_OPEN"

def parse_cowrie_file(fpath: Path) -> list:
    sessions = defaultdict(list)
    with open(fpath, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try:
                ev = json.loads(line)
                sid = ev.get("session","")
                if sid: sessions[sid].append(ev)
            except: continue
    return list(sessions.values())

def build_record(events: list) -> dict | None:
    if not events: return None
    events.sort(key=lambda e: e.get("timestamp",""))
    first, last = events[0], events[-1]

    def to_ts(s):
        try: return datetime.fromisoformat(s.replace("Z","+00:00")).timestamp()
        except: return 0.0

    t_start = to_ts(first.get("timestamp",""))
    t_end   = to_ts(last.get("timestamp",""))
    ts_list = [to_ts(e.get("timestamp","")) for e in events if e.get("timestamp")]

    cmds, logins, downloads, seq = [], 0, [], []
    t_first_auth = t_first_cmd = None

    for ev in events:
        etype = ev.get("eventid","")
        if etype in ("cowrie.login.failed","cowrie.login.success"):
            logins += 1
            seq.append("ACCESS_BRUTE_SSH")
            if t_first_auth is None: t_first_auth = to_ts(ev.get("timestamp",""))
        elif etype == "cowrie.command.input":
            cmd = ev.get("input","")
            cmds.append(cmd)
            seq.append(label_command(cmd))
            if t_first_cmd is None: t_first_cmd = to_ts(ev.get("timestamp",""))
        elif etype == "cowrie.session.file_download":
            downloads.append(ev.get("url",""))
            seq.append("EXEC_WGET_EXEC")
        elif etype == "cowrie.session.file_upload":
            seq.append("EXFIL_SCP_DATA")

    if not seq: seq = ["RECON_IP_SCAN"]
    meaningful = [s for s in seq if not s.startswith("RECON")]
    label = meaningful[-1] if meaningful else seq[-1]

    return {
        "session_id":           first.get("session",""),
        "src_ip":               first.get("src_ip",""),
        "src_port":             first.get("src_port",0),
        "dst_port":             22,
        "protocol":             "ssh",
        "t_start":              t_start,
        "t_end":                t_end,
        "t_first_auth":         t_first_auth or t_start,
        "t_first_cmd":          t_first_cmd  or t_start,
        "event_timestamps":     ts_list,
        "session_duration_s":   max(0.0, t_end - t_start),
        "n_events":             len(events),
        "n_commands":           len(cmds),
        "login_attempts":       logins,
        "command_text":         " ; ".join(cmds) if cmds else "[no commands]",
        "n_downloads":          len(downloads),
        "bytes_in":             sum(e.get("size",0) for e in events if "size" in e),
        "bytes_out":            0,
        "micro_state":          label,
        "micro_state_sequence": ",".join(seq),
        "source":               "cowrie_real",
        "associated_cve":       "",
    }

In [3]:
# ── Parse all Cowrie logs ──
all_records = []
log_files   = sorted(COWRIE_LOG_DIR.glob("cowrie.json*"))
if SMOKE:
    log_files = log_files[:1]          # one day of capture is enough to smoke-test
print(f"Found {len(log_files)} Cowrie log files")

if not log_files:
    print("⚠  No Cowrie logs found in", COWRIE_LOG_DIR)
    print("   Place your cowrie.json.YYYY-MM-DD files there and re-run.")
else:
    for fpath in log_files:
        sessions = parse_cowrie_file(fpath)
        recs = [build_record(s) for s in sessions]
        recs = [r for r in recs if r is not None]
        all_records.extend(recs)
        print(f"  {fpath.name}: {len(recs)} sessions")

    df_cowrie = pd.DataFrame(all_records).drop_duplicates("session_id")
    print(f"\nTotal Cowrie sessions: {len(df_cowrie):,}")
    print("\nMicro-state distribution:")
    print(df_cowrie["micro_state"].value_counts().head(20))
    df_cowrie.to_parquet(DATA_PROC / "real_cowrie_raw.parquet", index=False)
    print(f"\nSaved → {DATA_PROC}/real_cowrie_raw.parquet")

Found 14 Cowrie log files


  cowrie.json: 7724 sessions
  cowrie.json.2026-11-01: 51 sessions


  cowrie.json.2026-11-02: 4370 sessions


  cowrie.json.2026-11-03: 5986 sessions


  cowrie.json.2026-11-04: 5380 sessions


  cowrie.json.2026-11-05: 4792 sessions
  cowrie.json.2026-11-06: 671 sessions


... (5 outputs left out)

  cowrie.json.2026-11-12: 9555 sessions


  cowrie.json.2026-11-13: 12046 sessions



Total Cowrie sessions: 114,490

Micro-state distribution:
micro_state
ACCESS_BRUTE_SSH      52748
DISC_ENV_PROBE        45709
RECON_IP_SCAN          8894
EXEC_SHELL_OPEN        6996
EXEC_WGET_EXEC          100
EXFIL_SCP_DATA           36
EVASION_HIST_ERASE        6
DISC_PROC_ENUM            1
Name: count, dtype: int64



Saved → C:\Users\mahit\OneDrive\Desktop\adaptive-honeypot-ml-CAPSTONE\honeypot_dataset\data\processed/real_cowrie_raw.parquet


## 1.2  Load CIC-IDS2017 Transfer Dataset

In [4]:
# ── CIC-IDS2017 label → micro-state mapping ──
CIC_LABEL_MAP = {
    # Benign is NOT a kill-chain state, so it is not one of the 45. It is
    # carried through feature extraction as label 45 and split off in
    # notebook 05 for the two-stage gate (DATA_INTEGRITY.md section 5).
    "BENIGN":              "BENIGN",
    "DoS Hulk":            "DISC_ENV_PROBE",
    "PortScan":            "RECON_IP_SCAN",
    "DDoS":                "DISC_NETSTAT_SCAN",
    "DoS GoldenEye":       "DISC_ENV_PROBE",
    "FTP-Patator":         "ACCESS_BRUTE_SSH",
    "SSH-Patator":         "ACCESS_BRUTE_SSH",
    "DoS slowloris":       "DISC_ENV_PROBE",
    "DoS Slowhttptest":    "DISC_ENV_PROBE",
    "Bot":                 "PERSIST_CRONTAB",
    "Web Attack Brute Force": "ACCESS_BRUTE_HTTP",
    "Web Attack XSS":      "EXEC_CURL_BASH",
    "Infiltration":        "LATERAL_SSH_SPREAD",
    "Web Attack Sql Injection": "EXEC_SHELL_OPEN",
    "Heartbleed":          "ACCESS_KEX_EXPLOIT",
}

CIC_DIR = DATA_RAW / "cic_ids2017"
cic_files = list(CIC_DIR.glob("*.csv")) + list(CIC_DIR.glob("*.parquet"))
if SMOKE:
    cic_files = cic_files[:1]

if not cic_files:
    print("⚠  CIC-IDS2017 files not found in", CIC_DIR)
    print("   Download from: https://www.unb.ca/cic/datasets/ids-2017.html")
    df_cic = pd.DataFrame()
else:
    frames = []
    for f in cic_files:
        chunk = pd.read_csv(f, low_memory=False) if f.suffix==".csv" else pd.read_parquet(f)
        chunk.columns = [c.strip() for c in chunk.columns]
        frames.append(chunk)
    df_cic_raw = pd.concat(frames, ignore_index=True)
    label_col  = next((c for c in df_cic_raw.columns if "label" in c.lower()), None)
    if label_col:
        # CIC-IDS2017's "Web Attack" labels are saved with a mis-encoded separator
        # (reads back as U+FFFD) instead of a plain space, e.g. "Web Attack � Brute Force".
        # Normalize before mapping so it matches "Web Attack Brute Force" etc.
        clean_labels = (df_cic_raw[label_col].astype(str)
                         .str.replace("�", " ", regex=False)
                         .str.replace(r"\s+", " ", regex=True)
                         .str.strip())
        df_cic_raw["micro_state"] = clean_labels.map(CIC_LABEL_MAP)
        df_cic = df_cic_raw[df_cic_raw["micro_state"].notna()].copy()
        df_cic["source"] = "cic_ids2017"
        _bn = df_cic["micro_state"] == "BENIGN"
        if BENIGN_CAP and _bn.sum() > BENIGN_CAP:
            keep = df_cic[_bn].sample(BENIGN_CAP, random_state=42)
            df_cic = pd.concat([df_cic[~_bn], keep], ignore_index=True)
            print(f"BENIGN capped to {BENIGN_CAP:,} rows")
        if SMOKE and len(df_cic) > SMOKE:
            per = max(1, SMOKE // df_cic["micro_state"].nunique())
            df_cic = (df_cic.sample(frac=1, random_state=42)
                        .groupby("micro_state").head(per))
        print(f"CIC sessions after mapping: {len(df_cic):,}")
        print(df_cic["micro_state"].value_counts())
    else:
        print("Could not find label column in CIC dataset")
        df_cic = pd.DataFrame()

BENIGN capped to 100,000 rows
CIC sessions after mapping: 657,646
micro_state
DISC_ENV_PROBE        252661
RECON_IP_SCAN         158930
DISC_NETSTAT_SCAN     128027
BENIGN                100000
ACCESS_BRUTE_SSH       13835
PERSIST_CRONTAB         1966
ACCESS_BRUTE_HTTP       1507
EXEC_CURL_BASH           652
LATERAL_SSH_SPREAD        36
EXEC_SHELL_OPEN           21
ACCESS_KEX_EXPLOIT        11
Name: count, dtype: int64


In [5]:
# ── 1.2b  Load UNSW-NB15 Transfer Dataset ──
UNSW_LABEL_MAP = {
    "Normal":          "BENIGN",               # see CIC_LABEL_MAP above
    "Fuzzers":         "DISC_ENV_PROBE",
    "Analysis":        "RECON_VULN_SCAN",
    "Backdoor":        "PERSIST_BACKDOOR_ADD",
    "DoS":             "DISC_NETSTAT_SCAN",
    "Exploits":        "ACCESS_KEX_EXPLOIT",
    "Generic":         "EXEC_SHELL_OPEN",
    "Reconnaissance":  "RECON_IP_SCAN",
    "Shellcode":       "EXEC_MEMFD_EXEC",
    "Worms":           "LATERAL_SSH_SPREAD",
}

UNSW_DIR = DATA_RAW / "unsw_nb15"
unsw_train = UNSW_DIR / "UNSW_NB15_training-set.csv"
unsw_test  = UNSW_DIR / "UNSW_NB15_testing-set.csv"

if unsw_train.exists() and unsw_test.exists():
    df_unsw_raw = pd.concat([
        pd.read_csv(unsw_train),
        pd.read_csv(unsw_test),
    ], ignore_index=True)
    df_unsw_raw.columns = [c.strip() for c in df_unsw_raw.columns]

    label_col = next((c for c in df_unsw_raw.columns
                      if c.lower() in ("attack_cat","attack_category")), None)
    if label_col:
        df_unsw_raw["micro_state"] = df_unsw_raw[label_col].str.strip().map(UNSW_LABEL_MAP)
        df_unsw = df_unsw_raw[df_unsw_raw["micro_state"].notna()].copy()
        df_unsw["source"] = "unsw_nb15"
        _bn = df_unsw["micro_state"] == "BENIGN"
        if BENIGN_CAP and _bn.sum() > BENIGN_CAP:
            keep = df_unsw[_bn].sample(BENIGN_CAP, random_state=42)
            df_unsw = pd.concat([df_unsw[~_bn], keep], ignore_index=True)
            print(f"BENIGN capped to {BENIGN_CAP:,} rows")
        if SMOKE and len(df_unsw) > SMOKE:
            per = max(1, SMOKE // df_unsw["micro_state"].nunique())
            df_unsw = (df_unsw.sample(frac=1, random_state=42)
                        .groupby("micro_state").head(per))
        print(df_unsw["micro_state"].value_counts())
    else:
        print("Could not find attack_cat column in UNSW dataset")
        df_unsw = pd.DataFrame()
else:
    print(f"⚠  UNSW files not found in {UNSW_DIR}")
    df_unsw = pd.DataFrame()

micro_state
BENIGN                  93000
EXEC_SHELL_OPEN         58871
ACCESS_KEX_EXPLOIT      44525
DISC_ENV_PROBE          24246
DISC_NETSTAT_SCAN       16353
RECON_IP_SCAN           13987
RECON_VULN_SCAN          2677
PERSIST_BACKDOOR_ADD     2329
EXEC_MEMFD_EXEC          1511
LATERAL_SSH_SPREAD        174
Name: count, dtype: int64


## 1.3  Combine Real Data

In [6]:
real_frames = []

if "df_cowrie" in dir() and len(df_cowrie) > 0:
    real_frames.append(df_cowrie)

if "df_cic" in dir() and len(df_cic) > 0:
    df_cic = df_cic.reset_index(drop=True)
    n = len(df_cic)

    def describe_cic_flow(row):
        proto = row.get("Protocol", "TCP")
        duration = row.get("Flow Duration", 0)
        fwd_pkts = row.get("Total Fwd Packets", 0)
        bwd_pkts = row.get("Total Backward Packets", 0)
        dst_port = row.get("Destination Port", 0)
        # NO label in the text. It used to end "classified as {label}", which
        # wrote the answer into command_text -> DistilBERT -> features 76-105,
        # i.e. group D was a label oracle for 100% of netflow rows. Those are
        # the 13 genuinely-real classes, so every per-class number on them was
        # inflated. Measured 2026-11-19; the frozen dataset has the same leak.
        return (f"{proto} flow to port {dst_port}, duration {duration}us, "
                f"{fwd_pkts} forward packets, {bwd_pkts} backward packets")

    df_cic["command_text"] = df_cic.apply(describe_cic_flow, axis=1)

    # ---- REAL flow timing, not a fabricated one -------------------------
    # CIC-IDS2017 ships genuine `Flow IAT Mean/Std/Max/Min` in MICROseconds.
    # Verified 2026-11-24 on 225,745 rows: 0 non-finite, 2 negatives,
    # min<=mean<=max coherent in 100.00%, and 100% of flows carry >=2 packets so
    # an inter-arrival time genuinely exists. Passing these in seconds lets
    # `extract_temporal` report provenance `flow_stats` and compute iat_mean /
    # std / min / max / cv from MEASUREMENTS instead of inventing one gap.
    #
    # The sequence-only features (median, skew, kurtosis, bursts, sub-100ms and
    # over-30s fractions) stay NaN: a flow summary cannot supply them, and a
    # number there would be fabrication dressed as evidence.
    def _usec(col):
        if col not in df_cic.columns:
            return np.full(n, np.nan)
        v = pd.to_numeric(df_cic[col], errors="coerce").to_numpy(dtype=float)
        v = np.where(v < 0, np.nan, v)      # CIC emits a handful of -1 / -12
        return v / 1e6
    cic_iat_mean = _usec("Flow IAT Mean")
    cic_iat_std  = _usec("Flow IAT Std")
    cic_iat_min  = _usec("Flow IAT Min")
    cic_iat_max  = _usec("Flow IAT Max")

    cic_aligned = pd.DataFrame({
        "session_id":           [f"cic_{i}" for i in range(n)],
        "iat_mean_s":           cic_iat_mean,
        "iat_std_s":            cic_iat_std,
        "iat_min_s":            cic_iat_min,
        "iat_max_s":            cic_iat_max,
        "micro_state":          df_cic["micro_state"].to_numpy(),
        "micro_state_sequence": df_cic["micro_state"].to_numpy(),
        "source":               "cic_transfer",
        "session_duration_s":   df_cic.get("Flow Duration", pd.Series([1.0]*n)).to_numpy() / 1e6,
        "bytes_in":             df_cic.get("Total Fwd Packets", pd.Series([0]*n)).to_numpy(),
        "bytes_out":            df_cic.get("Total Backward Packets", pd.Series([0]*n)).to_numpy(),
        "n_events": 5, "n_commands": 0, "login_attempts": 0,
        "command_text": df_cic["command_text"].to_numpy(),
        "t_start": 1_700_000_000.0, "t_end": 1_700_000_001.0,
        "dst_port": 80, "protocol": "tcp", "associated_cve": "",
    })
    real_frames.append(cic_aligned)

if "df_unsw" in dir() and len(df_unsw) > 0:
    df_unsw = df_unsw.reset_index(drop=True)
    n = len(df_unsw)

    def describe_unsw_flow(row):
        proto = row.get("proto", "tcp")
        duration = row.get("dur", 0)
        sbytes = row.get("sbytes", 0)
        dbytes = row.get("dbytes", 0)
        dsport = row.get("dsport", 0)
        # No label in the text -- see describe_cic_flow above.
        return (f"{proto} flow to port {dsport}, duration {duration}s, "
                f"{sbytes} source bytes, {dbytes} dest bytes")

    df_unsw["command_text"] = df_unsw.apply(describe_unsw_flow, axis=1)

    # UNSW-NB15 train/test set has NO dsport column (that's only in the raw
    # 4-part files). Default to port 22 since most UNSW attack records
    # represent generic network traffic, not service-specific sessions.
    if "dsport" in df_unsw.columns:
        dst_port_col = pd.to_numeric(df_unsw["dsport"], errors="coerce").fillna(22).astype(int).to_numpy()
    else:
        dst_port_col = np.full(n, 22, dtype=int)

    proto_col = (df_unsw["proto"].to_numpy() if "proto" in df_unsw.columns
                else np.full(n, "tcp", dtype=object))

    # ---- UNSW: the MEAN is real, nothing else is -------------------------
    # `sinpkt` is the mean source inter-packet time in MILLIseconds -- confirmed
    # by dur/(spkts-1) vs sinpkt having a ratio of exactly 1000.0. 98.5% of flows
    # carry >=2 packets.
    #
    # `sjit` is deliberately NOT mapped to iat_std. Argus jitter is a MEAN
    # DEVIATION, not a standard deviation; calling it iat_std would launder one
    # statistic as another, which is the same class of error as the fabricated
    # IAT. So these rows report provenance `flow_mean_only` and leave std / min /
    # max NaN.
    if "sinpkt" in df_unsw.columns:
        _sp = pd.to_numeric(df_unsw["sinpkt"], errors="coerce").to_numpy(dtype=float)
        unsw_iat_mean = np.where(_sp > 0, _sp / 1e3, np.nan)
    else:
        unsw_iat_mean = np.full(n, np.nan)

    unsw_aligned = pd.DataFrame({
        "session_id":           [f"unsw_{i}" for i in range(n)],
        "iat_mean_s":           unsw_iat_mean,
        "micro_state":          df_unsw["micro_state"].to_numpy(),
        "micro_state_sequence": df_unsw["micro_state"].to_numpy(),
        "source":               "unsw_transfer",
        "session_duration_s":   df_unsw.get("dur",    pd.Series([1.0]*n)).to_numpy(),
        "bytes_in":             df_unsw.get("sbytes",  pd.Series([0]*n)).to_numpy(),
        "bytes_out":            df_unsw.get("dbytes",  pd.Series([0]*n)).to_numpy(),
        "n_events": 5, "n_commands": 0, "login_attempts": 0,
        "command_text": df_unsw["command_text"].to_numpy(),
        "t_start": 1_700_000_000.0, "t_end": 1_700_000_001.0,
        "dst_port": dst_port_col,
        "protocol": proto_col,
        "associated_cve": "",
    })
    real_frames.append(unsw_aligned)

# ---- Synthetic Cowrie sessions, generated as EVENTS and parsed ----------
# Real cowrie_logs/ only exercises 8-9 of the 45 micro-states, and the 13-day
# capture produced zero new ones over the 8-day capture -- collecting SSH for
# longer does not broaden the taxonomy (DATA_INTEGRITY.md 3d). The other ~36
# classes have to be generated, and cowrie_simulator does it by emitting
# Cowrie EVENTS through this same `build_record`, so every derived feature is
# computed rather than drawn. 0 of 97 arithmetic checks fail on its output.
#
# These rows are SYNTHETIC and tagged `simulated_cowrie_events`. Never merge
# them into a "real data" count -- that is the provenance mistake the 15,000
# old `cowrie_real` rows were (CLAUDE.md, DECISIONS.md 2026-08-28).
SIM_PER_CLASS = 16_000          # 45 x 16,000 = 720,000, TabSyn's old volume
if SMOKE:
    # at least 4 per class, or notebook 05's stratified split cannot hold a
    # class out into both val and test
    SIM_PER_CLASS = max(4, SMOKE // 45)
if SIM_PER_CLASS:
    from src.generators.cowrie_simulator import generate_balanced_sessions as gen_sim
    df_sim = gen_sim(0, seed=42, per_class=SIM_PER_CLASS)
    print(f"\nSynthetic Cowrie-event sessions: {len(df_sim):,} "
          f"({df_sim['micro_state'].nunique()} classes, "
          f"parser agreement {100*df_sim['label_agrees'].mean():.1f}%)")
    real_frames.append(df_sim)

if not real_frames:
    print("⚠  No real data found — generating minimal placeholder")
    from src.generators.kill_chain_simulator import generate_balanced_sessions
    df_real = generate_balanced_sessions(n_total=10_000, seed=42)
    df_real["source"] = "simulated_placeholder"
else:
    df_real = pd.concat(real_frames, ignore_index=True)

# `seen_before` feeds tls_host -> attacker_repeat_visitor. Only the Cowrie
# frames carry it, so after the concat it is NaN for the netflow rows -- and
# float(bool(nan)) is 1.0, which would mark every netflow row a repeat
# visitor. Recompute it for the whole corpus from src_ip instead, which is
# what the real path has to do anyway (cowrie_simulator.mark_repeat_visitors).
# Measured caveat: 99.06% of real sessions ARE repeat visits (1,071 IPs over
# 114,490 sessions), so this feature is near-constant whatever we do.
if "src_ip" not in df_real.columns:
    df_real["src_ip"] = None
df_real["seen_before"] = df_real["src_ip"].notna() & df_real.duplicated("src_ip")

# Temporal provenance, recorded per row so it survives into real_meta.parquet
# and the validator can check that no row presents an unmeasurable feature as a
# measurement. This is the audit trail for DATA_INTEGRITY.md Part 0.
from src.extractors.temporal import temporal_provenance
df_real["temporal_provenance"] = [temporal_provenance(r)
                                  for r in df_real.to_dict("records")]
print("\nTemporal provenance by source:")
print(pd.crosstab(df_real["source"], df_real["temporal_provenance"]).to_string())

print(f"\nTotal real sessions: {len(df_real):,}")
print("Sources:", df_real["source"].value_counts().to_dict())
print("Micro-states:", df_real["micro_state"].nunique(), "unique classes")
print("\ncommand_text.value_counts().head(10):")
print(df_real["command_text"].value_counts().head(10))
df_real.to_parquet(DATA_PROC / "real_sessions_combined.parquet", index=False)
print(f"\nSaved → {DATA_PROC}/real_sessions_combined.parquet")

2026-11-24 21:15:43,029 INFO Generated 720000 sessions from Cowrie events (45 classes); parser agreement 24.0%



Synthetic Cowrie-event sessions: 720,000 (45 classes, parser agreement 24.0%)



Temporal provenance by source:
temporal_provenance      event_stream  flow_mean_only  flow_stats  none
source                                                                 
cic_transfer                        0             289      657351     6
cowrie_real                    114489               0           0     1
simulated_cowrie_events        720000               0           0     0
unsw_transfer                       0          256891           0   782

Total real sessions: 1,749,809
Sources: {'simulated_cowrie_events': 720000, 'cic_transfer': 657646, 'unsw_transfer': 257673, 'cowrie_real': 114490}


Micro-states: 46 unique classes

command_text.value_counts().head(10):


command_text
[no commands]                                                                                      254885
uname -s -v -n -r -m                                                                                60698
echo xsec                                                                                           18800
TCP flow to port 80, duration 3us, 2 forward packets, 0 backward packets                            16245
udp flow to port 0, duration 9e-06s, 114 source bytes, 0 dest bytes                                 15266
wget "https://aws-yup-distr-04.gaijinent.com/wt_launcher_linux_1.0.3.50.tar.gz?distr=gi5yqwek3"     11635
TCP flow to port 80, duration 4us, 2 forward packets, 0 backward packets                            10964
TCP flow to port 80, duration 1us, 2 forward packets, 0 backward packets                             9235
udp flow to port 0, duration 8e-06s, 114 source bytes, 0 dest bytes                                  8972
udp flow to port 0, duration 3e-0


Saved → C:\Users\mahit\OneDrive\Desktop\adaptive-honeypot-ml-CAPSTONE\honeypot_dataset\data\processed/real_sessions_combined.parquet
